# Agent 智能体设计模式

## 路由（Routing）
路由会对输入进行分类，并将其定向到专门的后续任务。此工作流程允许分离关注点，并构建更专业的提示。如果没有此工作流程，针对一种输入进行优化可能会损害其他输入的性能。
![](https://i-blog.csdnimg.cn/direct/6e3a309fb70f4bf8a0c2ffd01559d7da.png)

In [1]:
from openai import OpenAI
from datetime import datetime
import json
from typing import List, Dict, Callable
import os
import re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
load_dotenv("/Users/a1-6/Documents/projects/DL/.env")
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

llm_name = llm_name = "qwen-plus"
def call_llm(user_prompt, system_prompt=""):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user","content": user_prompt}
        ]
    response = client.chat.completions.create(
            model= llm_name,
            messages = messages)
    return(response.choices[0].message.content)

In [2]:
def extract_xml(text: str, tag: str) -> str:
    match = re.search(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return match.group(1) if match else ""


In [3]:
writing_prompt={
"议论文":"""
请写一篇议论文，围绕以下主题展开明确的观点，并提供有力的论据支持。要求：
1．**观点鲜明**：开篇提出明确的论点。
2．**逻辑严谨**：通过事实、数据、案例进行论证。\
3．**结构清晰**：包括引言、论证、结论三部分。\
4．**语言有说服力**：条理分明，语言简洁有力。 
""",
"散文":"""
请写一篇散文，围绕以下主题抒发真实细腻的情感，描绘生动的画面。
要求：
1. **情感真挚**：用细腻的文字表达内心感受。
2. **意境优美**：通过比喻、拟人等修辞，营造氛围。
3. **语言流畅**：语言自然，富有节奏感。
4. **不拘结构**：可以自由组织段落和内容。
""",
"诗歌":
"""
请创作一首诗歌，围绕以下主题，用简洁生动的语言表达情感或描绘画面。要求：
1. **意象丰富**：用生动的意象表现主题。
2. **语言凝练**：短句精练，富有节奏感。
3. **情感浓烈**：真实自然地传达内心情感。
4. **形式灵活**：可自由选择格律诗或现代诗。
""",
"小说":"""
请创作一篇短篇小说，围绕以下主题展开生动的故事情节。要求：
1. **人物鲜明**：刻画有特点的人物形象。
2. **情节完整**：包含起因、发展、高潮和结局。
3. **描写生动**：通过坏境、动作、对话推动情节。
4. **主题突出**：故事紧扣主题，能引发思考。
"""
}

In [4]:
def route(input: str, routes: Dict[str, str]) -> str:
    """Route input to specialized prompt using content classification."""
    selector_prompt = f"""
    你是一个作家,请根据用户输入的主题,判断其需要的写作风格,并将内容路由到对应模块。
    输入的主题是{input}
    路由的分类包括:
    {list(routes.keys())}
    你需要用简洁的语言归纳给出分类的理由,然后输出分类名称。都需要输出到XML格式标签中。
    <reasoning>
    份类的理由
    </reasoning>
    <selection>
    分类名称
    </selection>
    """.strip()
    route_response = call_llm(selector_prompt)
    reasoning = extract_xml(route_response, 'reasoning')
    route_key = extract_xml(route_response, 'selection').strip().lower()
    print("分类的理由")
    print(reasoning)
    print(f"\n分类名称：{route_key}")


In [ ]:
requests = ["为什么学习编程", "天边的云"]

In [ ]:
for request in requests:
    print("-" * 40)
    print(request)
    print("-" * 40)
    response = route(request, writing_prompt)
    print(response)